# Drone Chord Swapper

MIDI chord drone with live-tweakable **velocity**, **strum time**, and **attack** via Tweakpane.
New chord triggers pick up whatever values the sliders currently show.

In [1]:
import { MidiAccess } from "@/midi/mod.ts"
import { createOSCClient } from "@/tools/osc.ts"
import { createTweakpane } from "@/tools/tweakpaneAdapter.ts"

function sleep(ms: number) {
  return new Promise((resolve) => setTimeout(resolve, ms))
}

// --- Tweakpane-controlled parameters ---
const params = {
  velocity: 100,
  strumMs: 250,
  attack: 0.7,
}

const pane = createTweakpane({ title: 'Drone Controls' })
pane.addBinding(params, 'velocity', { min: 0, max: 127, step: 1 })
pane.addBinding(params, 'strumMs',  { min: 0, max: 1000, step: 1, label: 'strum (ms)' })
pane.addBinding(params, 'attack',   { min: 0, max: 1, step: 0.01 })

console.log('Tweakpane params ready')

Tweakpane params ready


In [2]:
pane.show()
null

[tweakpane] Auto-initializing server...
[tweakpane] Server running at http://0.0.0.0:61620


null

In [3]:
// --- Chord definitions (MIDI note numbers) ---
const chords: Record<string, number[]> = {
  Cmaj:  [60, 64, 67],
  Dmin:  [62, 65, 69],
  Emin:  [64, 67, 71],
  Fmaj:  [65, 69, 72],
  Gmaj:  [67, 71, 74],
  Amin:  [69, 72, 76],
  Cmaj7: [60, 64, 67, 71],
  Dm9:   [62, 65, 69, 72, 76],
}

// --- Input note → chord mapping ---
const noteToChord: Record<number, string> = {
  36: 'Cmaj',
  37: 'Dmin',
  38: 'Emin',
  39: 'Fmaj',
  40: 'Gmaj',
  41: 'Amin',
  42: 'Cmaj7',
  43: 'Dm9',
}

console.log('Chords loaded:', Object.keys(chords).join(', '))

Chords loaded: Cmaj, Dmin, Emin, Fmaj, Gmaj, Amin, Cmaj7, Dm9


In [ ]:
// --- Config ---
const INPUT_NAME  = 'from Max 1'
const OUTPUT_NAME = 'IAC Driver Bus 1'
const OUTPUT_CHANNEL = 0

// --- Setup MIDI ---
const midi = MidiAccess.open()

const inputs = midi.listInputs()
console.log('MIDI inputs:', inputs.map(p => p.name))
const outputs = midi.listOutputs()
console.log('MIDI outputs:', outputs.map(p => p.name))

const inputPort  = inputs.find(p => p.name.includes(INPUT_NAME)) ?? inputs[0]
const outputInfo = outputs.find(p => p.name.includes(OUTPUT_NAME)) ?? outputs[0]

const input  = midi.openInput(inputPort.id, { rateHz: 200 })
const output = midi.openOutput(outputInfo.id)
const osc    = createOSCClient('127.0.0.1', 7099)

console.log(`Listening: "${inputPort.name}"  →  Output: "${outputInfo.name}"`)

// --- State ---
let currentChordNotes: number[] = []
let currentChordName: string | null = null
let swapping = false

const OCTAVE_RANGE = 2
const LOW_BOUND  = (notes: number[]) => Math.min(...notes) - 12 * OCTAVE_RANGE
const HIGH_BOUND = (notes: number[]) => Math.max(...notes) + 12 * OCTAVE_RANGE

function randomInversion(playing: number[], baseNotes: number[]): number[] {
  const lo = LOW_BOUND(baseNotes)
  const hi = HIGH_BOUND(baseNotes)
  const result = [...playing]
  const idx = Math.floor(Math.random() * result.length)
  const note = result[idx]
  const canGoUp   = note + 12 <= hi
  const canGoDown = note - 12 >= lo
  if (canGoUp && canGoDown) {
    result[idx] = Math.random() < 0.5 ? note + 12 : note - 12
  } else if (canGoUp) {
    result[idx] = note + 12
  } else if (canGoDown) {
    result[idx] = note - 12
  }
  return result
}

async function sendChord(notes: number[]) {
  const hadNotes = currentChordNotes.length > 0
  for (const note of currentChordNotes) {
    output.noteOff(OUTPUT_CHANNEL, note, 0)
  }
  currentChordNotes = []
  if (hadNotes) await sleep(10)

  // Read live tweakpane values
  osc.send('/attack', params.attack)

  for (let i = 0; i < notes.length; i++) {
    output.noteOn(OUTPUT_CHANNEL, notes[i], params.velocity)
    currentChordNotes.push(notes[i])
    if (params.strumMs > 0 && i < notes.length - 1) {
      await sleep(params.strumMs)
    }
  }
}

async function swapChord(newChordName: string) {
  if (swapping) return
  swapping = true
  const baseNotes = chords[newChordName]
  if (!baseNotes) { swapping = false; return }

  let newNotes: number[]
  if (currentChordName === newChordName) {
    newNotes = randomInversion(currentChordNotes, baseNotes)
    console.log(`Invert  ${newChordName}  ${JSON.stringify(newNotes)}`)
  } else {
    newNotes = [...baseNotes]
    console.log(`Chord → ${newChordName}  ${JSON.stringify(newNotes)}`)
  }

  await sendChord(newNotes)
  currentChordName = newChordName
  swapping = false
}

// --- Listen for input noteOn ---
input.onNoteOn((evt) => {
  const chordName = noteToChord[evt.noteNum]
  if (chordName) {
    swapChord(chordName)
  } else {
    console.log(`Unmapped note ${evt.noteNum} (vel=${evt.velocity})`)
  }
})

console.log('Chord swapper running — trigger chords via MIDI input')

MIDI inputs: [
  "IAC Driver Bus 1",
  "IAC Driver Bus 2",
  "IAC Driver Bus 3",
  "IAC Driver Bus 4",
  "IAC Driver Bus 5",
  "IAC Driver Bus 6",
  "IAC Driver Bus 7",
  "IAC Driver Bus 8",
  "from Max 1",
  "from Max 2"
]
MIDI outputs: [
  "IAC Driver Bus 1",
  "IAC Driver Bus 2",
  "IAC Driver Bus 3",
  "IAC Driver Bus 4",
  "IAC Driver Bus 5",
  "IAC Driver Bus 6",
  "IAC Driver Bus 7",
  "IAC Driver Bus 8",
  "to Max 1",
  "to Max 2"
]
Listening: "from Max 1"  →  Output: "IAC Driver Bus 1"
Chord swapper running — trigger chords via MIDI input


Unmapped note 47 (vel=26)
Chord → Amin  [69,72,76]
Invert  Amin  [69,84,76]
Invert  Amin  [57,84,76]
Invert  Amin  [57,84,64]
Chord → Cmaj  [60,64,67]
Invert  Cmaj  [60,64,55]
Invert  Cmaj  [60,64,67]
Chord → Emin  [64,67,71]
Chord → Gmaj  [67,71,74]
Invert  Gmaj  [67,83,74]
Invert  Gmaj  [67,83,62]
Invert  Gmaj  [67,83,50]
Invert  Gmaj  [79,83,50]
Chord → Emin  [64,67,71]
Chord → Gmaj  [67,71,74]
Invert  Gmaj  [79,71,74]
Chord → Emin  [64,67,71]
Chord → Amin  [69,72,76]
Invert  Amin  [57,72,76]
Chord → Gmaj  [67,71,74]
Chord → Cmaj  [60,64,67]
Chord → Dmin  [62,65,69]
Invert  Dmin  [62,53,69]
Chord → Emin  [64,67,71]
Chord → Cmaj  [60,64,67]


In [ ]:
// --- Cleanup: stop held notes and close ports ---
for (const note of currentChordNotes) {
  output.noteOff(OUTPUT_CHANNEL, note, 0)
}
currentChordNotes = []
input.close()
output.close()
osc.close()
midi.close()
pane.shutdown()
console.log('Cleaned up')